# 2. Reproduce the tobacco detector experiments

Train YOLOv8n/m/x on the frozen split, then evaluate selected checkpoints on validation and test images. All scientific code is defined below. Default execution explains the workflow without loading YOLO, downloading weights, or training. Install `requirements-experiments.txt` before opting in.

In [ ]:
from pathlib import Path
import sys
import os
import json
import tempfile

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'notebooks').is_dir() and (p / 'experiments/splits/items.csv').is_file()), None)
if ROOT is None:
    raise RuntimeError('Start Jupyter from the repository root or a subdirectory.')
if str(ROOT / 'notebooks') not in sys.path:
    sys.path.insert(0, str(ROOT / 'notebooks'))
from reproduction import (new_run, frozen_assignments, write_local_splits,
                          check_detector_results, capture_environment)
os.environ.setdefault('MPLCONFIGDIR', str(Path(tempfile.gettempdir()) / 'tobacco-notebook-mpl'))
os.environ.setdefault('YOLO_CONFIG_DIR', str(ROOT / 'experiments/.ultralytics'))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
RESULTS = ROOT / 'experiments/results'


## Choose inputs and a new output directory

Provide the original dataset in this layout:

```text
tobacco_yolo/
  images/<image-name>.jpg
  labels/<same-image-name>.txt
```

Each label row is `class_id x_center y_center width height`, with normalized coordinates. **Class 0 must mean tobacco**, not weeds. Software can check IDs but cannot determine their semantic meaning; confirm it from the annotation source.

Every new run uses a fresh directory under `experiments/reproductions/`. Reusing a run name raises an error to protect previous results. To retry a failed experiment, choose another name. No historical result is overwritten.

In [ ]:
RUN_TRAINING = False
CLASS_ZERO_CONFIRMED_TOBACCO = False
DATASET = ROOT / 'data/tobacco_yolo'  # Or Path('/your/external/tobacco_yolo')
RUN_NAME = 'detector-reproduction-01'
DEVICE = '0'  # CUDA GPU index; use 'cpu' only for an explicitly CPU experiment.
MODELS = ['yolov8n.pt', 'yolov8m.pt', 'yolov8x.pt']
EPOCHS, IMAGE_SIZE, BATCH_SIZE = 50, 1024, 14
run_dir = None
training_complete = False
print('Dataset present:', (DATASET / 'images').is_dir() and (DATASET / 'labels').is_dir())
print('Training enabled:', RUN_TRAINING)
display(pd.Series(json.loads((RESULTS / 'yolo/run_metadata.json').read_text())))


In [ ]:
import csv
import re
import shutil
import time
from collections import Counter
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}


## Read YOLO annotations

Each nonempty row contains class ID and four normalized box coordinates. Class 0 must mean tobacco; software cannot verify that semantic claim.

In [ ]:
def image_paths(image_dir: Path) -> list[Path]:
    return sorted(path for path in image_dir.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)

def parse_label(path: Path) -> tuple[list[int], list[str]]:
    class_ids: list[int] = []
    errors: list[str] = []
    if not path.exists():
        return class_ids, errors
    for line_number, raw_line in enumerate(path.read_text(encoding="utf-8").splitlines(), start=1):
        line = raw_line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) != 5:
            errors.append(f"{path}:{line_number}: expected 5 columns, found {len(parts)}")
            continue
        try:
            class_id = int(parts[0])
            values = [float(value) for value in parts[1:]]
        except ValueError:
            errors.append(f"{path}:{line_number}: non-numeric YOLO value")
            continue
        if class_id < 0:
            errors.append(f"{path}:{line_number}: negative class id {class_id}")
        for value in values:
            if value < 0 or value > 1:
                errors.append(f"{path}:{line_number}: normalized coordinate outside [0, 1]: {value}")
        class_ids.append(class_id)
    return class_ids, errors


## Audit the dataset

Count images, labels, boxes, malformed rows, and filename groups. Prefix groups are a leakage diagnostic, not proof of independent acquisition.

In [ ]:
def infer_prefixes(stems: list[str]) -> dict[str, int]:
    patterns = [
        re.compile(r"^(.*?)(?:[_-]?frame[_-]?\d+)$", re.IGNORECASE),
        re.compile(r"^(.*?)(?:[_-]?\d{3,})$"),
    ]
    counts: Counter[str] = Counter()
    for stem in stems:
        for pattern in patterns:
            match = pattern.match(stem)
            if match and match.group(1):
                counts[match.group(1).rstrip("_-")] += 1
                break
    return dict(counts.most_common(20))

def audit_dataset(dataset, output, expected_class_zero="tobacco", allow_missing_labels=False, group_regex=None):
    image_dir = dataset / "images"
    label_dir = dataset / "labels"
    failures: list[str] = []

    if not image_dir.exists():
        failures.append(f"missing image directory: {image_dir}")
    if not label_dir.exists():
        failures.append(f"missing label directory: {label_dir}")
    if failures:
        report = {
            "dataset": str(dataset),
            "expected_class_zero": expected_class_zero,
            "images": 0,
            "labels": 0,
            "boxes": 0,
            "class_counts": {},
            "missing_labels": [],
            "missing_labels_count": 0,
            "extra_label_files": [],
            "extra_label_files_count": 0,
            "malformed_rows": [],
            "malformed_rows_count": 0,
            "likely_frame_prefixes": {},
            "group_regex": group_regex,
            "group_regex_matches": {},
            "status": "fail",
            "failures": failures,
        }
        output.parent.mkdir(parents=True, exist_ok=True)
        output.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
        print("\n".join(failures))
        return report

    images = image_paths(image_dir)
    label_files = sorted(label_dir.glob("*.txt"))
    label_stems = {path.stem for path in label_files}
    image_stems = [path.stem for path in images]
    missing_labels: list[str] = []
    extra_labels = sorted(label_stems - set(image_stems))
    class_counts: Counter[int] = Counter()
    total_boxes = 0
    malformed: list[str] = []

    for image in images:
        label_path = label_dir / f"{image.stem}.txt"
        if not label_path.exists():
            missing_labels.append(image.stem)
            continue
        class_ids, errors = parse_label(label_path)
        malformed.extend(errors)
        class_counts.update(class_ids)
        total_boxes += len(class_ids)

    if not images:
        failures.append(f"no images found in {image_dir}")
    if missing_labels and not allow_missing_labels:
        failures.append(f"{len(missing_labels)} images are missing label files")
    if malformed:
        failures.append(f"{len(malformed)} malformed label rows")
    if total_boxes == 0:
        failures.append("no bounding boxes found")
    if 0 not in class_counts and total_boxes > 0:
        failures.append("class id 0 is absent, but class 0 is expected to be tobacco/crop")

    group_matches: Counter[str] = Counter()
    if group_regex:
        regex = re.compile(group_regex)
        for stem in image_stems:
            match = regex.search(stem)
            if match:
                group_matches[match.group(1)] += 1

    report = {
        "dataset": str(dataset),
        "expected_class_zero": expected_class_zero,
        "images": len(images),
        "labels": len(label_files),
        "boxes": total_boxes,
        "class_counts": dict(sorted(class_counts.items())),
        "missing_labels": missing_labels[:100],
        "missing_labels_count": len(missing_labels),
        "extra_label_files": extra_labels[:100],
        "extra_label_files_count": len(extra_labels),
        "malformed_rows": malformed[:100],
        "malformed_rows_count": len(malformed),
        "likely_frame_prefixes": infer_prefixes(image_stems),
        "group_regex": group_regex,
        "group_regex_matches": dict(group_matches.most_common(20)),
        "status": "fail" if failures else "ok",
        "failures": failures,
    }
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
    print(json.dumps(report, indent=2))
    return report


## Preserve the original split

The saved manifests contain Windows paths. We read their filenames portably, verify them against `items.csv`, and later write absolute paths for this machine. We **do not generate a new random split**. Expected membership is 78 train, 16 validation, 16 test.

The split seed `20260627` controls dataset splitting; it is not a recorded training seed. The original runner relies on framework defaults for training randomness and augmentations. Its full resolved training arguments are not included here. Capturing the new environment makes differences auditable, but exact numerical replication is not guaranteed.

The saved manifest digest includes path strings and box counts, not image bytes. It is not a dataset-content checksum. Counts and filenames cannot establish that two datasets are identical; obtain archive checksums from the dataset owner when available. The split is image-based and does not establish independence by field or acquisition session.

In [ ]:
assignments = frozen_assignments(ROOT)
display(pd.DataFrame({'split': assignments.keys(), 'images': [len(v) for v in assignments.values()]}))


## Validate inputs and prepare a fresh run

A mismatch stops reproduction. Inspect the report before training. Environment capture is infrastructure; dataset checks are visible above.

In [ ]:
if RUN_TRAINING:
    if not CLASS_ZERO_CONFIRMED_TOBACCO:
        raise ValueError('Confirm class 0 means tobacco before enabling training.')
    DATASET = DATASET.expanduser().resolve()
    if not (DATASET / 'images').is_dir() or not (DATASET / 'labels').is_dir():
        raise FileNotFoundError('Supply the external images/ and labels/ directories first.')
    import torch
    if DEVICE != 'cpu' and not torch.cuda.is_available():
        raise RuntimeError('CUDA is unavailable. Install a compatible PyTorch/driver environment first.')
    run_dir = new_run(ROOT, RUN_NAME)
    audit_dataset(DATASET, run_dir / 'dataset_preflight.json')
    preflight = json.loads((run_dir / 'dataset_preflight.json').read_text())
    assert preflight['status'] == 'ok'
    assert preflight['images'] == 110 and preflight['boxes'] == 4436
    assert preflight['extra_label_files_count'] == 0
    assert set(preflight['class_counts']) == {'0'}, 'Expected single-class tobacco annotations.'
    items = pd.read_csv(ROOT / 'experiments/splits/items.csv')
    for item in items.itertuples():
        label = DATASET / 'labels' / f'{item.stem}.txt'
        assert sum(bool(line.strip()) for line in label.read_text().splitlines()) == item.boxes
    local_splits = write_local_splits(ROOT, DATASET, run_dir / 'splits')
    capture_environment(run_dir / 'environment')
else:
    print('Skipped data preflight: set DATASET, confirm class meaning, and enable RUN_TRAINING to continue.')


## Describe the data and collect artifacts

The generated YAML points to relocated manifests. Preserve training-log metrics separately from held-out evaluation metrics.

In [ ]:
def write_data_yaml(dataset, splits, output, class_name="tobacco") -> Path:
    yaml_path = output / "data.generated.yaml"
    yaml_path.write_text(
        "\n".join(
            [
                f"path: {json.dumps(str(dataset.resolve()))}",
                f"train: {json.dumps(str((splits / 'train.txt').resolve()))}",
                f"val: {json.dumps(str((splits / 'val.txt').resolve()))}",
                f"test: {json.dumps(str((splits / 'test.txt').resolve()))}",
                "names:",
                f"  0: {class_name}",
                "",
            ]
        ),
        encoding="utf-8",
    )
    return yaml_path

def collect_metrics(run_dir: Path) -> dict[str, str]:
    results_csv = run_dir / "results.csv"
    if not results_csv.exists():
        return {}
    with results_csv.open("r", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    if not rows:
        return {}
    return {key.strip(): value for key, value in rows[-1].items()}

def flatten_metrics(prefix: str, metrics: object) -> dict[str, str]:
    output: dict[str, str] = {}
    results_dict = getattr(metrics, "results_dict", None)
    if isinstance(results_dict, dict):
        for key, value in results_dict.items():
            output[f"{prefix}_{key}".replace(" ", "_")] = str(value)
    speed = getattr(metrics, "speed", None)
    if isinstance(speed, dict):
        for key, value in speed.items():
            output[f"{prefix}_speed_{key}_ms"] = str(value)
    return output

def model_file_size_mb(path: Path) -> str:
    if not path.exists():
        return ""
    return f"{path.stat().st_size / (1024 * 1024):.3f}"

def parameter_count(detector: object) -> str:
    model = getattr(detector, "model", None)
    if model is None:
        return ""
    try:
        return str(sum(parameter.numel() for parameter in model.parameters()))
    except Exception:
        return ""

def copy_artifacts(run_dir: Path, output: Path, model_name: str) -> None:
    for filename in ["results.csv", "results.png", "confusion_matrix.png", "PR_curve.png", "F1_curve.png"]:
        source = run_dir / filename
        if source.exists():
            shutil.copy2(source, output / f"{model_name}_{filename}")


## Train and evaluate the selected checkpoints

The calls below expose the actual SGD settings and evaluation protocol. Each model records failures and the final completeness check rejects partial results. Selection uses best.pt, falling back to last.pt if best.pt is absent; both paths are recorded. Test results must not guide hyperparameter tuning.

In [ ]:
def evaluate_checkpoint(best_weights, last_weights, data_yaml, output, run_name, row,
                        model_factory, imgsz=1024, batch=14, device="0"):
    eval_detector = model_factory(str(best_weights if best_weights.exists() else last_weights))
    row["trained_parameters"] = parameter_count(eval_detector)
    for split in ["val", "test"]:
        try:
            split_metrics = eval_detector.val(
                data=str(data_yaml),
                split=split,
                imgsz=imgsz,
                batch=batch,
                device=device,
                plots=(split == "val"),
                project=str(output.resolve()),
                name=f"{run_name}_{split}",
                exist_ok=True,
            )
            row.update(flatten_metrics(split, split_metrics))
        except Exception as exc:
            row[f"{split}_status"] = "failed"
            row[f"{split}_error"] = str(exc)


### Train one model

The training call explicitly specifies the historical optimizer and schedule. After fitting, collect the final log row, select a checkpoint, and evaluate it. Failures remain visible in the comparison table.

In [ ]:
def train_one_model(model, data_yaml, output, model_factory, epochs=50, imgsz=1024, batch=14, device="0"):
    model_name = Path(model).stem
    run_name = model_name
    start = time.time()
    run_dir = output / run_name
    row = {
        "model": model,
        "model_name": model_name,
        "run_dir": str(run_dir),
        "epochs": str(epochs),
        "imgsz": str(imgsz),
        "batch": str(batch),
        "device": str(device),
        "status": "started",
        "notes": "",
    }
    try:
        print(f"Training {model}", flush=True)
        detector = model_factory(model)
        row["initial_parameters"] = parameter_count(detector)
        train_metrics = detector.train(
            data=str(data_yaml),
            epochs=epochs,
            imgsz=imgsz,
            batch=batch,
            device=device,
            optimizer="SGD",
            lr0=0.005,
            momentum=0.937,
            weight_decay=0.0005,
            cos_lr=True,
            amp=True,
            plots=True,
            project=str(output.resolve()),
            name=run_name,
            exist_ok=True,
        )
        elapsed = time.time() - start
        row["train_seconds"] = f"{elapsed:.2f}"
        row.update(flatten_metrics("train_return", train_metrics))

        metrics = collect_metrics(run_dir)
        row.update(metrics)
        weights_dir = run_dir / "weights"
        best_weights = weights_dir / "best.pt"
        last_weights = weights_dir / "last.pt"
        row["best_weights"] = str(best_weights) if best_weights.exists() else ""
        row["best_weights_mb"] = model_file_size_mb(best_weights)
        row["last_weights"] = str(last_weights) if last_weights.exists() else ""

        evaluate_checkpoint(best_weights, last_weights, data_yaml, output, run_name,
                            row, model_factory, imgsz, batch, device)
        row["status"] = "ok"
        copy_artifacts(run_dir, output, model_name)
    except Exception as exc:
        elapsed = time.time() - start
        row["train_seconds"] = f"{elapsed:.2f}"
        row["status"] = "failed"
        row["notes"] = str(exc)
        copy_artifacts(run_dir, output, model_name)

    return row


### Compare models and save the experiment record

Apply the same settings to every model, export comparison rows, and record the run configuration. A subsequent completeness check rejects failed or missing evaluations.

In [ ]:
def train_detectors(dataset, splits, models, output, epochs=50, imgsz=1024, batch=14, device="0", class_name="tobacco", model_factory=None):
    Path(os.environ["YOLO_CONFIG_DIR"]).mkdir(parents=True, exist_ok=True)
    if model_factory is None:
        from ultralytics import YOLO
        model_factory = YOLO

    output.mkdir(parents=True, exist_ok=True)
    data_yaml = write_data_yaml(dataset, splits, output, class_name)

    comparison_rows: list[dict[str, str]] = []
    for model in models:
        row = train_one_model(model, data_yaml, output, model_factory, epochs, imgsz, batch, device)
        comparison_rows.append(row)

    fieldnames = sorted({key for row in comparison_rows for key in row})
    with (output / "model_comparison.csv").open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(comparison_rows)

    metadata = {
        "dataset": str(dataset),
        "splits": str(splits),
        "models": models,
        "epochs": epochs,
        "imgsz": imgsz,
        "batch": batch,
        "device": device,
        "model_status": {row["model_name"]: row["status"] for row in comparison_rows},
    }
    (output / "run_metadata.json").write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")

    return comparison_rows


## Run the comparison

For GPU memory errors, reduce batch size consistently for every model and choose a fresh run name. Report changes and CPU timing explicitly. Restart the kernel before retrying a failed experiment.

In [ ]:
if RUN_TRAINING:
    if run_dir is None:
        raise RuntimeError('Run validation first.')
    train_detectors(DATASET, local_splits, MODELS, run_dir / 'yolo',
                    epochs=EPOCHS, imgsz=IMAGE_SIZE, batch=BATCH_SIZE, device=DEVICE)
    check_detector_results(run_dir / 'yolo/model_comparison.csv')
    training_complete = True
else:
    print('Training skipped; no new metrics were generated.')


## Compare the new measurements with the historical run

Differences can arise from framework versions, randomness, GPU kernels, and hardware. Do not substitute historical timing for the new machine's timing. Report changed settings together with the captured environment. Test data should not be used to tune hyperparameters.

In [ ]:
if training_complete:
    old = pd.read_csv(RESULTS / 'yolo/model_comparison.csv').set_index('model_name')
    new = pd.read_csv(run_dir / 'yolo/model_comparison.csv').set_index('model_name')
    keys = ['test_metrics/precision(B)', 'test_metrics/recall(B)',
            'test_metrics/mAP50(B)', 'test_metrics/mAP50-95(B)', 'test_speed_inference_ms']
    display(pd.concat({'Historical': old[keys], 'Reproduction': new[keys]}, axis=1))
    print('Outputs:', run_dir)
    print('Use this trained model in notebook 3:', run_dir / 'yolo/yolov8x/weights/best.pt')
else:
    print('No new experiment was run; there are no new metrics to compare.')
